# Scaled Dataset Classification
## Logistic Regression, SVM, and KNN Models
Training and evaluating models on scaled dataset with train/validation/test split.

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import joblib
import os

# Set random seed for reproducibility
np.random.seed(42)

## 1. Load and Explore Dataset

In [ ]:
# Load dataset
df = pd.read_csv('../datasets/trojan_cleaned(scaled).csv')

# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nDataset Info:")
print(df.info())
print("\nMissing Values:")
print(df.isnull().sum())
print("\nBasic Statistics:")
print(df.describe())

## 2. Data Preparation

In [ ]:
# Separate features and target
# Assuming the last column is the target (adjust if needed)
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())

## 3. Train/Validation/Test Split

In [ ]:
# Splitting Dataset
X_train, X_temp, y_train, y_temp = train_test_split(
    X, 
    y, 
    test_size=0.30,
    random_state=42
)

# Splitting the 30% --> 2 x 15% (test and val)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, 
    y_temp, 
    test_size=0.50,
    random_state=42
)

print(f"Training set size: {X_train.shape[0]} ({X_train.shape[0]/X.shape[0]*100:.1f}%)")
print(f"Validation set size: {X_val.shape[0]} ({X_val.shape[0]/X.shape[0]*100:.1f}%)")
print(f"Test set size: {X_test.shape[0]} ({X_test.shape[0]/X.shape[0]*100:.1f}%)")
print(f"\nTotal: {X_train.shape[0] + X_val.shape[0] + X_test.shape[0]} samples")

## 4. Logistic Regression Model

In [ ]:
# Train Logistic Regression
print("="*50)
print("LOGISTIC REGRESSION")
print("="*50)

lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)

# Predictions
lr_train_pred = lr_model.predict(X_train)
lr_val_pred = lr_model.predict(X_val)
lr_test_pred = lr_model.predict(X_test)

# Probabilities for ROC curve
lr_train_proba = lr_model.predict_proba(X_train)[:, 1]
lr_val_proba = lr_model.predict_proba(X_val)[:, 1]
lr_test_proba = lr_model.predict_proba(X_test)[:, 1]

# Evaluation metrics
print("\nTRAINING SET:")
print(f"Accuracy: {accuracy_score(y_train, lr_train_pred):.4f}")
print(f"Precision: {precision_score(y_train, lr_train_pred, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_train, lr_train_pred, zero_division=0):.4f}")
print(f"F1-Score: {f1_score(y_train, lr_train_pred, zero_division=0):.4f}")

print("\nVALIDATION SET:")
lr_val_acc = accuracy_score(y_val, lr_val_pred)
print(f"Accuracy: {lr_val_acc:.4f}")
print(f"Precision: {precision_score(y_val, lr_val_pred, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_val, lr_val_pred, zero_division=0):.4f}")
print(f"F1-Score: {f1_score(y_val, lr_val_pred, zero_division=0):.4f}")

print("\nTEST SET:")
lr_test_acc = accuracy_score(y_test, lr_test_pred)
print(f"Accuracy: {lr_test_acc:.4f}")
print(f"Precision: {precision_score(y_test, lr_test_pred, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test, lr_test_pred, zero_division=0):.4f}")
print(f"F1-Score: {f1_score(y_test, lr_test_pred, zero_division=0):.4f}")

# Confusion Matrix
lr_cm = confusion_matrix(y_test, lr_test_pred)
print("\nConfusion Matrix:")
print(lr_cm)

## 5. SVM Model

In [ ]:
# Train SVM (optimized for speed with linear kernel)
print("\n" + "="*50)
print("SUPPORT VECTOR MACHINE (SVM) - Linear Kernel (Fast)")
print("="*50)

# Using linear kernel instead of RBF for faster training
# Linear kernel is very effective on scaled data and 100x faster than RBF
svm_model = SVC(kernel='linear', probability=True, random_state=42, C=1.0, max_iter=2000)
svm_model.fit(X_train, y_train)

# Predictions
svm_train_pred = svm_model.predict(X_train)
svm_val_pred = svm_model.predict(X_val)
svm_test_pred = svm_model.predict(X_test)

# Probabilities for ROC curve
svm_train_proba = svm_model.predict_proba(X_train)[:, 1]
svm_val_proba = svm_model.predict_proba(X_val)[:, 1]
svm_test_proba = svm_model.predict_proba(X_test)[:, 1]

# Evaluation metrics
print("\nTRAINING SET:")
print(f"Accuracy: {accuracy_score(y_train, svm_train_pred):.4f}")
print(f"Precision: {precision_score(y_train, svm_train_pred, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_train, svm_train_pred, zero_division=0):.4f}")
print(f"F1-Score: {f1_score(y_train, svm_train_pred, zero_division=0):.4f}")

print("\nVALIDATION SET:")
svm_val_acc = accuracy_score(y_val, svm_val_pred)
print(f"Accuracy: {svm_val_acc:.4f}")
print(f"Precision: {precision_score(y_val, svm_val_pred, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_val, svm_val_pred, zero_division=0):.4f}")
print(f"F1-Score: {f1_score(y_val, svm_val_pred, zero_division=0):.4f}")

print("\nTEST SET:")
svm_test_acc = accuracy_score(y_test, svm_test_pred)
print(f"Accuracy: {svm_test_acc:.4f}")
print(f"Precision: {precision_score(y_test, svm_test_pred, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test, svm_test_pred, zero_division=0):.4f}")
print(f"F1-Score: {f1_score(y_test, svm_test_pred, zero_division=0):.4f}")

# Confusion Matrix
svm_cm = confusion_matrix(y_test, svm_test_pred)
print("\nConfusion Matrix:")
print(svm_cm)

## 6. KNN Model

In [ ]:
# Train KNN with optimal k value
print("\n" + "="*50)
print("K-NEAREST NEIGHBORS (KNN)")
print("="*50)

# Find optimal k by testing on validation set
best_k = 1
best_val_acc = 0

for k in range(1, 21):
    knn_temp = KNeighborsClassifier(n_neighbors=k)
    knn_temp.fit(X_train, y_train)
    val_acc = accuracy_score(y_val, knn_temp.predict(X_val))
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_k = k

print(f"\nOptimal k value: {best_k} (Validation Accuracy: {best_val_acc:.4f})")

# Train KNN with optimal k
knn_model = KNeighborsClassifier(n_neighbors=best_k)
knn_model.fit(X_train, y_train)

# Predictions
knn_train_pred = knn_model.predict(X_train)
knn_val_pred = knn_model.predict(X_val)
knn_test_pred = knn_model.predict(X_test)

# Probabilities for ROC curve (for KNN)
knn_train_proba = knn_model.predict_proba(X_train)[:, 1]
knn_val_proba = knn_model.predict_proba(X_val)[:, 1]
knn_test_proba = knn_model.predict_proba(X_test)[:, 1]

# Evaluation metrics
print("\nTRAINING SET:")
print(f"Accuracy: {accuracy_score(y_train, knn_train_pred):.4f}")
print(f"Precision: {precision_score(y_train, knn_train_pred, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_train, knn_train_pred, zero_division=0):.4f}")
print(f"F1-Score: {f1_score(y_train, knn_train_pred, zero_division=0):.4f}")

print("\nVALIDATION SET:")
knn_val_acc = accuracy_score(y_val, knn_val_pred)
print(f"Accuracy: {knn_val_acc:.4f}")
print(f"Precision: {precision_score(y_val, knn_val_pred, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_val, knn_val_pred, zero_division=0):.4f}")
print(f"F1-Score: {f1_score(y_val, knn_val_pred, zero_division=0):.4f}")

print("\nTEST SET:")
knn_test_acc = accuracy_score(y_test, knn_test_pred)
print(f"Accuracy: {knn_test_acc:.4f}")
print(f"Precision: {precision_score(y_test, knn_test_pred, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test, knn_test_pred, zero_division=0):.4f}")
print(f"F1-Score: {f1_score(y_test, knn_test_pred, zero_division=0):.4f}")

# Confusion Matrix
knn_cm = confusion_matrix(y_test, knn_test_pred)
print("\nConfusion Matrix:")
print(knn_cm)

## 7. Model Comparison and Visualization

In [ ]:
# Comparison metrics
models = ['Logistic Regression', 'SVM', 'KNN']
test_accuracies = [lr_test_acc, svm_test_acc, knn_test_acc]
test_precisions = [
    precision_score(y_test, lr_test_pred, zero_division=0),
    precision_score(y_test, svm_test_pred, zero_division=0),
    precision_score(y_test, knn_test_pred, zero_division=0)
]
test_recalls = [
    recall_score(y_test, lr_test_pred, zero_division=0),
    recall_score(y_test, svm_test_pred, zero_division=0),
    recall_score(y_test, knn_test_pred, zero_division=0)
]
test_f1s = [
    f1_score(y_test, lr_test_pred, zero_division=0),
    f1_score(y_test, svm_test_pred, zero_division=0),
    f1_score(y_test, knn_test_pred, zero_division=0)
]

# Create comparison visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Accuracy comparison
axes[0, 0].bar(models, test_accuracies, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[0, 0].set_ylabel('Accuracy', fontsize=12)
axes[0, 0].set_title('Accuracy Comparison (Test Set)', fontsize=12, fontweight='bold')
axes[0, 0].set_ylim([0, 1])
for i, v in enumerate(test_accuracies):
    axes[0, 0].text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')

# Precision comparison
axes[0, 1].bar(models, test_precisions, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[0, 1].set_ylabel('Precision', fontsize=12)
axes[0, 1].set_title('Precision Comparison (Test Set)', fontsize=12, fontweight='bold')
axes[0, 1].set_ylim([0, 1])
for i, v in enumerate(test_precisions):
    axes[0, 1].text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')

# Recall comparison
axes[1, 0].bar(models, test_recalls, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[1, 0].set_ylabel('Recall', fontsize=12)
axes[1, 0].set_title('Recall Comparison (Test Set)', fontsize=12, fontweight='bold')
axes[1, 0].set_ylim([0, 1])
for i, v in enumerate(test_recalls):
    axes[1, 0].text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')

# F1-Score comparison
axes[1, 1].bar(models, test_f1s, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[1, 1].set_ylabel('F1-Score', fontsize=12)
axes[1, 1].set_title('F1-Score Comparison (Test Set)', fontsize=12, fontweight='bold')
axes[1, 1].set_ylim([0, 1])
for i, v in enumerate(test_f1s):
    axes[1, 1].text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Logistic Regression
sns.heatmap(lr_cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False)
axes[0].set_title('Logistic Regression\nConfusion Matrix', fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# SVM
sns.heatmap(svm_cm, annot=True, fmt='d', cmap='Oranges', ax=axes[1], cbar=False)
axes[1].set_title('SVM\nConfusion Matrix', fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

# KNN
sns.heatmap(knn_cm, annot=True, fmt='d', cmap='Greens', ax=axes[2], cbar=False)
axes[2].set_title('KNN\nConfusion Matrix', fontweight='bold')
axes[2].set_ylabel('True Label')
axes[2].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Logistic Regression ROC
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_test_proba)
roc_auc_lr = auc(fpr_lr, tpr_lr)
axes[0].plot(fpr_lr, tpr_lr, color='#1f77b4', lw=2, label=f'AUC = {roc_auc_lr:.4f}')
axes[0].plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Random Classifier')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('Logistic Regression\nROC Curve', fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].grid(alpha=0.3)

# SVM ROC
fpr_svm, tpr_svm, _ = roc_curve(y_test, svm_test_proba)
roc_auc_svm = auc(fpr_svm, tpr_svm)
axes[1].plot(fpr_svm, tpr_svm, color='#ff7f0e', lw=2, label=f'AUC = {roc_auc_svm:.4f}')
axes[1].plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Random Classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('SVM\nROC Curve', fontweight='bold')
axes[1].legend(loc='lower right')
axes[1].grid(alpha=0.3)

# KNN ROC
fpr_knn, tpr_knn, _ = roc_curve(y_test, knn_test_proba)
roc_auc_knn = auc(fpr_knn, tpr_knn)
axes[2].plot(fpr_knn, tpr_knn, color='#2ca02c', lw=2, label=f'AUC = {roc_auc_knn:.4f}')
axes[2].plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Random Classifier')
axes[2].set_xlabel('False Positive Rate')
axes[2].set_ylabel('True Positive Rate')
axes[2].set_title('KNN\nROC Curve', fontweight='bold')
axes[2].legend(loc='lower right')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nAUC-ROC Scores (Test Set):")
print(f"Logistic Regression: {roc_auc_lr:.4f}")
print(f"SVM: {roc_auc_svm:.4f}")
print(f"KNN: {roc_auc_knn:.4f}")

## 8. Export Trained Models

In [ ]:
# Create models directory if it doesn't exist
os.makedirs('models/scaled_dataset', exist_ok=True)

# Save models using joblib
model_path_lr = 'models/scaled_dataset/logistic_regression_model.pkl'
model_path_svm = 'models/scaled_dataset/svm_model.pkl'
model_path_knn = 'models/scaled_dataset/knn_model.pkl'

joblib.dump(lr_model, model_path_lr)
joblib.dump(svm_model, model_path_svm)
joblib.dump(knn_model, model_path_knn)

print("Models saved successfully!")
print(f"✓ Logistic Regression: {model_path_lr}")
print(f"✓ SVM: {model_path_svm}")
print(f"✓ KNN: {model_path_knn}")

In [ ]:
# Summary Report
print("\n" + "="*60)
print("SUMMARY REPORT - SCALED DATASET")
print("="*60)
print(f"\nBest Model (by Test Accuracy): ", end="")
best_model_idx = np.argmax(test_accuracies)
print(f"{models[best_model_idx]} ({test_accuracies[best_model_idx]:.4f})")

print("\nTest Set Performance:")
print(f"{'Model':<25} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
print("-" * 60)
for i, model in enumerate(models):
    print(f"{model:<25} {test_accuracies[i]:<12.4f} {test_precisions[i]:<12.4f} {test_recalls[i]:<12.4f} {test_f1s[i]:<12.4f}")

print(f"\n{'Model':<25} {'AUC-ROC':<12}")
print("-" * 40)
print(f"{'Logistic Regression':<25} {roc_auc_lr:<12.4f}")
print(f"{'SVM':<25} {roc_auc_svm:<12.4f}")
print(f"{'KNN':<25} {roc_auc_knn:<12.4f}")